# Group B — Operator Verification

Verifies that each of the 8 ASCAL operators produces the mathematically correct result
after **every single demonstration**. Assertions are derived directly from the operator
definitions in `src/ascal/algorithm.py`.

### After a positive demo `(pre, action, post)` — let `delta = post − pre`

| Operator | Formula | Property to assert |
|---|---|---|
| **ULP** | `L_pre ← {h ∩ pre \| h ∈ L_pre}` | `L_pre hypothesis ⊆ pre_state` |
| **RUP** | `U_pre ← {h ∈ U_pre \| h ⊆ pre}` | `∀ hp ∈ U_pre : hp ⊆ pre_state` |
| **ULE** | `L_eff ← {h ∪ delta \| h ∈ L_eff}` | `delta ⊆ L_eff hypothesis` |
| **RLE** | `L_eff ← {h ∈ L_eff \| h ⊆ post}` | `∀ hL ∈ L_eff : hL ⊆ post_state` |
| **RUE** | `U_eff ← {h ∈ U_eff \| delta ⊆ h}` | `∀ hU ∈ U_eff : delta ⊆ hU` |
| **UUE** | `U_eff ← {h ∩ post \| h ∈ U_eff}` | `∀ hU ∈ U_eff : hU ⊆ post_state` |

### After a negative demo `(pre, action, None)`

| Operator | Formula | Property to assert |
|---|---|---|
| **RLP** | `L_pre ← {h ∈ L_pre \| h ⊄ pre}` | `∀ hL ∈ L_pre : hL ⊄ pre_state` |
| **UUP** | Extend firing hypotheses with a distinguishing literal | `∀ hU ∈ U_pre : hU ⊄ pre_state` |

Reference: `AlgorithmExplained.md §13 Group B`

In [1]:
import sys, os, tempfile, shutil

sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

from unified_planning.shortcuts import *
from unified_planning.io import PDDLReader
import unified_planning
get_environment().credits_stream = None

from ascal.learner import Learner
from ascal.models import Literal, State, Action, Demonstration
from ascal.transitions import generate_lifted_demonstrations_from_problem

## Setup

In [2]:
MOCKUP_DIR   = os.path.join('..', 'benchmarks', 'mockup')
DOMAIN_FILE  = os.path.join(MOCKUP_DIR, 'domain.pddl')
PROBLEM_FILE = os.path.join(MOCKUP_DIR, 'problems', 'problem-00.pddl')

assert os.path.exists(DOMAIN_FILE),  f'Domain not found: {DOMAIN_FILE}'
assert os.path.exists(PROBLEM_FILE), f'Problem not found: {PROBLEM_FILE}'
print(f'Domain:  {DOMAIN_FILE}')
print(f'Problem: {PROBLEM_FILE}')

Domain:  ../benchmarks/mockup/domain.pddl
Problem: ../benchmarks/mockup/problems/problem-00.pddl


In [3]:
reader  = PDDLReader()
problem = reader.parse_problem(DOMAIN_FILE, PROBLEM_FILE)

all_fluents    = problem.fluents
all_actions    = problem.actions
static_fluents = problem.get_static_fluents()
action_names   = [a.name for a in all_actions]

print('Actions :', action_names)
print('Fluents :', [f.name for f in all_fluents])
print('Static  :', [f.name for f in static_fluents])

Actions : ['pickup']
Fluents : ['on', 'on_table', 'clear', 'arm_empty', 'holding']
Static  : ['on']


## Helpers

### Boundary snapshot

We capture a shallow copy of all four boundaries **before** calling `learner.update(demo)`.
`frozenset` objects are immutable so a shallow copy of the outer `set` is sufficient.

In [4]:
def snapshot(learner):
    """Capture current boundaries as a plain dict of sets (safe to compare after mutation)."""
    return {
        'L_pre': {n: set(learner.L_pre[n]) for n in action_names},
        'U_pre': {n: set(learner.U_pre[n]) for n in action_names},
        'L_eff': {n: set(learner.L_eff[n]) for n in action_names},
        'U_eff': {n: set(learner.U_eff[n]) for n in action_names},
    }

def fmt_lits(fset):
    """Format a frozenset of Literals as a short sorted string."""
    if not fset:
        return '∅'
    return '{' + ', '.join(sorted(str(l) for l in fset)) + '}'

print('Helpers defined.')

Helpers defined.


### Check functions

Each function receives the **learner after learning** and the **demo** that was processed.
For negative demos, it also needs `pre_b` (the snapshot taken **before** learning) to detect
whether UUP actually had any firing hypotheses to extend.

In [5]:
def check_group_b_positive(learner, demo, pre_b):
    """Assert all 6 positive-demo operator postconditions.

    Returns a list of violation strings (empty = all OK).
    """
    n          = demo.action.name
    pre_state  = demo.pre_state.literals
    post_state = demo.post_state.literals
    delta      = post_state.difference(pre_state)
    violations = []

    # ── ULP ──────────────────────────────────────────────────────────────────
    # Formula: L_pre ← {h ∩ pre | h ∈ L_pre_before}
    # Expected: current L_pre hypothesis ⊆ pre_state
    hL_pre = next(iter(learner.L_pre[n]))
    if not hL_pre.issubset(pre_state):
        extra = hL_pre - pre_state
        violations.append(
            f'ULP ✗  L_pre ⊄ pre_state\n'
            f'       spurious literals: {fmt_lits(extra)}')

    # ── RUP ──────────────────────────────────────────────────────────────────
    # Formula: U_pre ← {h ∈ U_pre | h ⊆ pre}
    # Expected: every remaining U_pre hypothesis ⊆ pre_state
    for hp in learner.U_pre[n]:
        if not hp.issubset(pre_state):
            extra = hp - pre_state
            violations.append(
                f'RUP ✗  U_pre hypothesis ⊄ pre_state\n'
                f'       hypothesis: {fmt_lits(hp)}\n'
                f'       spurious:   {fmt_lits(extra)}')

    # ── ULE ──────────────────────────────────────────────────────────────────
    # Formula: L_eff ← {h ∪ delta | h ∈ L_eff_before}
    # Expected: current L_eff hypothesis ⊇ delta
    hL_eff = next(iter(learner.L_eff[n]))
    if not delta.issubset(hL_eff):
        missing = delta - hL_eff
        violations.append(
            f'ULE ✗  delta ⊄ L_eff hypothesis\n'
            f'       delta:   {fmt_lits(delta)}\n'
            f'       missing: {fmt_lits(missing)}')

    # ── RLE ──────────────────────────────────────────────────────────────────
    # Formula: L_eff ← {h ∈ L_eff | h ⊆ post}
    # Expected: every remaining L_eff hypothesis ⊆ post_state
    for hL in learner.L_eff[n]:
        if not hL.issubset(post_state):
            extra = hL - post_state
            violations.append(
                f'RLE ✗  L_eff hypothesis ⊄ post_state\n'
                f'       hypothesis: {fmt_lits(hL)}\n'
                f'       spurious:   {fmt_lits(extra)}')

    # ── RUE ──────────────────────────────────────────────────────────────────
    # Formula: U_eff ← {h ∈ U_eff | delta ⊆ h}
    # Expected: every remaining U_eff hypothesis ⊇ delta
    for hU in learner.U_eff[n]:
        if not delta.issubset(hU):
            missing = delta - hU
            violations.append(
                f'RUE ✗  delta ⊄ U_eff hypothesis\n'
                f'       hypothesis: {fmt_lits(hU)}\n'
                f'       missing:    {fmt_lits(missing)}')

    # ── UUE ──────────────────────────────────────────────────────────────────
    # Formula: U_eff ← {h ∩ post | h ∈ U_eff_before}
    # Expected: every remaining U_eff hypothesis ⊆ post_state
    for hU in learner.U_eff[n]:
        if not hU.issubset(post_state):
            extra = hU - post_state
            violations.append(
                f'UUE ✗  U_eff hypothesis ⊄ post_state\n'
                f'       hypothesis: {fmt_lits(hU)}\n'
                f'       spurious:   {fmt_lits(extra)}')

    return violations


def check_group_b_negative(learner, demo, pre_b):
    """Assert both negative-demo operator postconditions.

    pre_b: boundary snapshot taken BEFORE learner.update(demo) was called.
    Returns a list of violation strings (empty = all OK).
    """
    n         = demo.action.name
    pre_state = demo.pre_state.literals
    violations = []

    # ── RLP ──────────────────────────────────────────────────────────────────
    # Formula: L_pre ← {h ∈ L_pre | h ⊄ pre}
    # Expected: no remaining L_pre hypothesis ⊆ pre_state
    for hL in learner.L_pre[n]:
        if hL.issubset(pre_state):
            violations.append(
                f'RLP ✗  L_pre hypothesis still ⊆ pre_state (should have been removed)\n'
                f'       hypothesis: {fmt_lits(hL)}')

    # ── UUP ──────────────────────────────────────────────────────────────────
    # Formula: For each hu ∈ U_pre that fires (hu ⊆ pre), extend with a
    #          literal from some hL ∈ L_pre that was absent from pre.
    # Expected: no remaining U_pre hypothesis ⊆ pre_state
    # NOTE: only asserted when at least one U_pre hypothesis fired BEFORE
    #       UUP ran (otherwise UUP had nothing to extend and is a no-op).
    fired_before = [h for h in pre_b['U_pre'][n] if h.issubset(pre_state)]
    if fired_before:
        for hU in learner.U_pre[n]:
            if hU.issubset(pre_state):
                violations.append(
                    f'UUP ✗  U_pre hypothesis still ⊆ pre_state (should have been extended)\n'
                    f'       hypothesis: {fmt_lits(hU)}\n'
                    f'       pre_state:  {fmt_lits(pre_state)}')
    else:
        pass  # UUP was a no-op — nothing to check

    return violations

print('check_group_b_positive() and check_group_b_negative() defined.')

check_group_b_positive() and check_group_b_negative() defined.


## Generate Demonstrations

In [6]:
def generate_demos(problem):
    """Delegates to :func:`ascal.transitions.generate_lifted_demonstrations_from_problem`.

    ``max_neg_per_step=0`` means no per-step cap (all failing groundings kept).
    """
    return generate_lifted_demonstrations_from_problem(
        problem,
        max_neg_per_step=0,
        max_check_per_action=None,
        seed=0,
        verbose=True,
    )



tmpdir = tempfile.mkdtemp(prefix='ascal_testB_')
orig_cwd = os.getcwd()
os.chdir(tmpdir)
try:
    pos_demos, neg_demos = generate_demos(problem)
finally:
    os.chdir(orig_cwd)
    shutil.rmtree(tmpdir, ignore_errors=True)

Plan: ['pickup(b2)']
Generated: 1 positive, 0 negative


## Main Learning Loop — Operator Checks After Every Demo

For each demo:
1. Snapshot boundaries **before** learning
2. Call `learner.update(demo)`  
3. Call the appropriate check function
4. Print a one-line status; stop immediately on any violation

In [7]:
learner = Learner(all_fluents, all_actions, static_fluents)

# Build interleaved sequence
if len(pos_demos) == 0:
    all_demos = list(neg_demos)
elif len(neg_demos) == 0:
    all_demos = list(pos_demos)
else:
    slice_size = len(neg_demos) / len(pos_demos)
    all_demos  = []
    for i, pos in enumerate(pos_demos):
        all_demos.extend(neg_demos[int(slice_size * i):int(slice_size * (i + 1))])
        all_demos.append(pos)

print(f'Sequence: {len(all_demos)} demos '
      f'({sum(1 for d in all_demos if d.post_state)} pos, '
      f'{sum(1 for d in all_demos if not d.post_state)} neg)')
print(f'Order: [{" ".join("POS" if d.post_state else "NEG" for d in all_demos)}]\n')

passed = failed = 0

for i, demo in enumerate(all_demos):
    kind  = 'POS' if demo.post_state is not None else 'NEG'
    label = f'demo {i:3d} [{kind}] {demo.action.name}'

    pre_b = snapshot(learner)          # capture state BEFORE learning
    learner.update(demo)                # update boundaries

    if demo.post_state is not None:
        violations = check_group_b_positive(learner, demo, pre_b)
    else:
        violations = check_group_b_negative(learner, demo, pre_b)

    # Size delta summary
    n = demo.action.name
    delta_str = (f'L_pre {len(next(iter(pre_b["L_pre"][n])))}→{len(next(iter(learner.L_pre[n])))}  '
                 f'U_pre_hyps {len(pre_b["U_pre"][n])}→{len(learner.U_pre[n])}  '
                 f'L_eff {len(next(iter(pre_b["L_eff"][n])))}→{len(next(iter(learner.L_eff[n])))}  '
                 f'U_eff {len(next(iter(pre_b["U_eff"][n])))}→{len(next(iter(learner.U_eff[n])))}')

    if violations:
        failed += 1
        print(f'❌ {label}')
        for v in violations:
            print(f'   {v}')
        print(f'   Sizes: {delta_str}')
        break
    else:
        passed += 1
        print(f'✓  {label}  |  {delta_str}')

print(f'\n{"=" * 60}')
if failed == 0:
    print(f'✅ ALL {passed} operator checks PASSED.')
else:
    print(f'❌ FAILED after {passed} passing checks.')

Sequence: 1 demos (1 pos, 0 neg)
Order: [POS]

✓  demo   0 [POS] pickup  |  L_pre 8→4  U_pre_hyps 1→1  L_eff 0→4  U_eff 8→4

✅ ALL 1 operator checks PASSED.


## Scale-up: Blocks Domain (4 actions, 20 problems)

Re-runs the same operator checks on the full blocks benchmark.
Demos are accumulated across all 20 problems before learning begins
(same methodology as the main evaluation pipeline).

In [8]:
BLOCKS_DIR     = os.path.join('..', 'benchmarks', 'blocks')
BLOCKS_DOMAIN  = os.path.join(BLOCKS_DIR, 'domain_extended.pddl')
BLOCKS_PROBS   = os.path.join(BLOCKS_DIR, 'problems')

assert os.path.exists(BLOCKS_DOMAIN), f'Blocks domain not found: {BLOCKS_DOMAIN}'
print(f'Blocks domain: {BLOCKS_DOMAIN}')

problem_files = sorted(f for f in os.listdir(BLOCKS_PROBS) if f.endswith('.pddl'))
print(f'Problem files: {len(problem_files)}')

# Parse schema from first problem (all share the same domain)
blocks_reader  = PDDLReader()
blocks_problem = blocks_reader.parse_problem(
    BLOCKS_DOMAIN, os.path.join(BLOCKS_PROBS, problem_files[0]))

b_fluents    = blocks_problem.fluents
b_actions    = blocks_problem.actions
b_static     = blocks_problem.get_static_fluents()
b_action_names = [a.name for a in b_actions]
print(f'Actions: {b_action_names}')

Blocks domain: ../benchmarks/blocks/domain_extended.pddl
Problem files: 21
Actions: ['pickup', 'putdown', 'stack', 'unstack']


In [9]:
# Collect all demonstrations across all problems
all_pos, all_neg = [], []

for pf in problem_files:
    prob = blocks_reader.parse_problem(BLOCKS_DOMAIN, os.path.join(BLOCKS_PROBS, pf))
    tmpdir = tempfile.mkdtemp(prefix='ascal_testB_blocks_')
    orig_cwd = os.getcwd()
    os.chdir(tmpdir)
    try:
        # Redefine action_names locally for generate_demos
        _orig_action_names = action_names
        action_names_backup = b_action_names  # not used in generate_demos, just for clarity
        p, n = generate_demos(prob)
    finally:
        os.chdir(orig_cwd)
        shutil.rmtree(tmpdir, ignore_errors=True)
    all_pos.extend(p)
    all_neg.extend(n)
    print(f'  {pf}: +{len(p)} pos  -{len(n)} neg')

# Deduplicate
all_pos = list(set(all_pos))
all_neg = list(set(all_neg))
print(f'\nTotal unique: {len(all_pos)} pos, {len(all_neg)} neg')

Plan: ['unstack(b8, b3, b1)', 'putdown(b8, b1, b2)', 'unstack(b3, b4, b1)', 'putdown(b3, b1, b2)', 'unstack(b7, b2, b1)', 'putdown(b7, b1, b2)', 'unstack(b2, b6, b1)', 'putdown(b2, b1, b3)', 'unstack(b6, b5, b1)', 'putdown(b6, b1, b2)', 'unstack(b5, b1, b2)', 'stack(b5, b4, b1)', 'pickup(b1, b2, b3)', 'stack(b1, b3, b2)', 'pickup(b6, b1, b2)', 'stack(b6, b5, b1)', 'pickup(b2, b1, b3)', 'stack(b2, b6, b1)', 'pickup(b7, b1, b2)', 'stack(b7, b2, b1)']
Generated: 20 positive, 24672 negative
  problem-00.pddl: +20 pos  -24672 neg
Plan: ['unstack(b5, b3, b1)', 'putdown(b5, b1, b2)', 'unstack(b7, b6, b1)', 'stack(b7, b5, b1)', 'unstack(b3, b2, b1)', 'putdown(b3, b1, b2)', 'unstack(b2, b1, b3)', 'putdown(b2, b1, b3)', 'pickup(b6, b1, b2)', 'stack(b6, b3, b1)', 'unstack(b4, b8, b1)', 'stack(b4, b1, b2)', 'unstack(b7, b5, b1)', 'putdown(b7, b1, b2)', 'pickup(b5, b1, b2)', 'stack(b5, b2, b1)', 'pickup(b7, b1, b2)', 'stack(b7, b5, b1)', 'unstack(b4, b1, b2)', 'putdown(b4, b1, b2)', 'pickup(b1, b2,

In [10]:
# Override action_names for check functions to use blocks actions
action_names = b_action_names

blocks_learner = Learner(b_fluents, b_actions, b_static)

# Build interleaved sequence
slice_size  = len(all_neg) / len(all_pos) if all_pos else 0
blocks_demos = []
for i, pos in enumerate(all_pos):
    blocks_demos.extend(all_neg[int(slice_size * i):int(slice_size * (i + 1))])
    blocks_demos.append(pos)

print(f'Sequence: {len(blocks_demos)} demos '
      f'({sum(1 for d in blocks_demos if d.post_state)} pos, '
      f'{sum(1 for d in blocks_demos if not d.post_state)} neg)\n')

passed = failed = 0

for i, demo in enumerate(blocks_demos):
    kind  = 'POS' if demo.post_state is not None else 'NEG'
    label = f'demo {i:4d} [{kind}] {demo.action.name}'

    pre_b = snapshot(blocks_learner)
    blocks_learner.update(demo)

    if demo.post_state is not None:
        violations = check_group_b_positive(blocks_learner, demo, pre_b)
    else:
        violations = check_group_b_negative(blocks_learner, demo, pre_b)

    if violations:
        failed += 1
        print(f'❌ {label}')
        for v in violations:
            print(f'   {v}')
        break
    else:
        passed += 1
        if i % 50 == 0:   # only print every 50th to avoid flooding
            n_str = demo.action.name
            print(f'✓  {label}  '
                  f'[L_pre={len(next(iter(blocks_learner.L_pre[n_str])))} '
                  f'U_pre_hyps={len(blocks_learner.U_pre[n_str])} '
                  f'L_eff={len(next(iter(blocks_learner.L_eff[n_str])))} '
                  f'U_eff={len(next(iter(blocks_learner.U_eff[n_str])))}]')

print(f'\n{"=" * 60}')
if failed == 0:
    print(f'✅ ALL {passed} operator checks PASSED on blocks domain.')
    print(f'   Converged: {blocks_learner.converged}')
    print(f'   Collapsed: {blocks_learner.collapsed_actions or "none"}')
else:
    print(f'❌ FAILED at demo {passed}.')

Sequence: 1759 demos (67 pos, 1692 neg)

✓  demo    0 [NEG] putdown  [L_pre=32 U_pre_hyps=16 L_eff=0 U_eff=32]
✓  demo   50 [NEG] unstack  [L_pre=32 U_pre_hyps=197 L_eff=0 U_eff=32]
✓  demo  100 [NEG] putdown  [L_pre=32 U_pre_hyps=337 L_eff=0 U_eff=32]
✓  demo  150 [NEG] putdown  [L_pre=16 U_pre_hyps=29 L_eff=4 U_eff=16]
✓  demo  200 [NEG] pickup  [L_pre=10 U_pre_hyps=1 L_eff=4 U_eff=10]
✓  demo  250 [NEG] stack  [L_pre=14 U_pre_hyps=4 L_eff=5 U_eff=14]
✓  demo  300 [NEG] stack  [L_pre=13 U_pre_hyps=2 L_eff=5 U_eff=13]
✓  demo  350 [NEG] stack  [L_pre=13 U_pre_hyps=1 L_eff=5 U_eff=13]
✓  demo  400 [NEG] unstack  [L_pre=12 U_pre_hyps=2 L_eff=5 U_eff=12]
✓  demo  450 [NEG] unstack  [L_pre=12 U_pre_hyps=2 L_eff=5 U_eff=12]
✓  demo  500 [NEG] stack  [L_pre=13 U_pre_hyps=1 L_eff=5 U_eff=13]
✓  demo  550 [POS] pickup  [L_pre=10 U_pre_hyps=1 L_eff=4 U_eff=10]
✓  demo  600 [NEG] stack  [L_pre=13 U_pre_hyps=1 L_eff=5 U_eff=13]
✓  demo  650 [NEG] unstack  [L_pre=12 U_pre_hyps=1 L_eff=5 U_eff=12]

## Final Boundary State — Blocks Domain

In [11]:
print('=== Final boundaries — blocks domain ===')
print(blocks_learner)

for a in b_actions:
    n = a.name
    hL_pre = next(iter(blocks_learner.L_pre[n]))
    hL_eff = next(iter(blocks_learner.L_eff[n]))
    hU_eff = next(iter(blocks_learner.U_eff[n]))

    print(f'\nAction: {n}')
    print(f'  L_pre ({len(hL_pre)} lits): {sorted(str(l) for l in hL_pre)}')
    print(f'  U_pre ({len(blocks_learner.U_pre[n])} hyps): '
          + str([sorted(str(l) for l in h) for h in blocks_learner.U_pre[n]]))
    print(f'  L_eff ({len(hL_eff)} lits): {sorted(str(l) for l in hL_eff)}')
    print(f'  U_eff ({len(hU_eff)} lits): {sorted(str(l) for l in hU_eff)}')
    print(f'  Converged: {blocks_learner.version_space_size[n]["converged"]}')

=== Final boundaries — blocks domain ===
Learner(actions=4, demos=1759, collapsed=0, converged=False)

Action: pickup
  L_pre (10 lits): ['arm_empty()', 'clear(x)', 'on_table(x)', '¬holding(x)', '¬holding(y)', '¬holding(z)', '¬on(x, y)', '¬on(x, z)', '¬on(y, x)', '¬on(z, x)']
  U_pre (1 hyps): [['arm_empty()', 'clear(x)', 'on_table(x)']]
  L_eff (4 lits): ['holding(x)', '¬arm_empty()', '¬clear(x)', '¬on_table(x)']
  U_eff (10 lits): ['holding(x)', '¬arm_empty()', '¬clear(x)', '¬holding(y)', '¬holding(z)', '¬on(x, y)', '¬on(x, z)', '¬on(y, x)', '¬on(z, x)', '¬on_table(x)']
  Converged: False

Action: putdown
  L_pre (10 lits): ['holding(x)', '¬arm_empty()', '¬clear(x)', '¬holding(y)', '¬holding(z)', '¬on(x, y)', '¬on(x, z)', '¬on(y, x)', '¬on(z, x)', '¬on_table(x)']
  U_pre (1 hyps): [['holding(x)']]
  L_eff (4 lits): ['arm_empty()', 'clear(x)', 'on_table(x)', '¬holding(x)']
  U_eff (10 lits): ['arm_empty()', 'clear(x)', 'on_table(x)', '¬holding(x)', '¬holding(y)', '¬holding(z)', '¬on(x